In [3]:
# =====================================================================
# 02 — Data Pipeline and Cross-Validation Splits
# Public Benchmark Release Setup (Leakage-Free Patient-Level Stratification)
# =====================================================================

import os
import re
import random
import hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# ─── 1. GLOBAL CONFIGURATION & PATH REFACTORING ──────────────────────
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)

# FIXED: Points directly to your 'processed' folder containing CC/ and MLO/
PROC_ROOT = "./processed"  
SPLIT_DIR = "./splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

N_FOLDS = 5
CLASS_FOLDERS = {
    'Benign': {'cc': 'CC/Benign', 'mlo': 'MLO/Benign'},
    'Malign': {'cc': 'CC/Malign', 'mlo': 'MLO/Malign'},
    'Normal': {'cc': 'CC/Normal', 'mlo': 'MLO/Normal'},
}

CLASS_NAMES = ['Benign', 'Malign', 'Normal']
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
VALID_EXTS = ('.png', '.jpg', '.jpeg')
PID_RE = re.compile(r'_?(\d{6,})')

def extract_pid(fname):
    """Extracts unique patient identifiers from filenames to guarantee patient-disjoint splits."""
    m = PID_RE.search(os.path.splitext(fname)[0])
    return m.group(1) if m else os.path.splitext(fname)[0]

# ─── 2. PATIENT-LEVEL TABLE COHORT ASSEMBLY ───────────────────────────
rows = []
for cls, fol in CLASS_FOLDERS.items():
    cc_dir = os.path.join(PROC_ROOT, fol['cc'])
    mlo_dir = os.path.join(PROC_ROOT, fol['mlo'])
    
    if not (os.path.isdir(cc_dir) and os.path.isdir(mlo_dir)):
        raise FileNotFoundError(f"Subdirectories missing in {PROC_ROOT}. Verify preprocessing stage outputs.")
        
    cc_files = {extract_pid(f): os.path.join(fol['cc'], f) for f in os.listdir(cc_dir) if f.lower().endswith(VALID_EXTS)}
    mlo_files = {extract_pid(f): os.path.join(fol['mlo'], f) for f in os.listdir(mlo_dir) if f.lower().endswith(VALID_EXTS)}
    
    # Enforce precise paired cross-view matching per patient side
    matched_pids = sorted(set(cc_files) & set(mlo_files))
    print(f"{cls:<7s}: Paired Patients Matched = {len(matched_pids):4d}")
    
    for pid in matched_pids:
        rows.append({
            'patient_id': pid,
            'class_3': cls,
            'label_3': class_to_idx[cls],
            'cc_path': cc_files[pid],
            'mlo_path': mlo_files[pid]
        })

df = pd.DataFrame(rows).sort_values('patient_id').reset_index(drop=True)

# Define standard downstream binary task targets
df['label_det'] = (df['class_3'] == 'Malign').astype(int)
df['label_char'] = np.where(df['class_3'] == 'Normal', -1, (df['class_3'] == 'Malign').astype(int))

# ─── 3. INTEGRITY & LEAKAGE CONTROL AUDIT ─────────────────────────────
duplicated_patients = df['patient_id'][df['patient_id'].duplicated()].tolist()
assert not duplicated_patients, f"CRITICAL LEAKAGE: Patient IDs map to multiple cohorts -> {duplicated_patients}"
print("✔ Leakage Integrity Check: PASSED (Mutual exclusivity verified across classes)")

# ─── 4. STRATIFIED PATIENT-LEVEL CROSS-VALIDATION SPLITTING ───────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
df['fold'] = -1

for fold_idx, (_, val_idx) in enumerate(skf.split(df, df['label_3'])):
    df.loc[val_idx, 'fold'] = fold_idx

assert (df['fold'] >= 0).all(), "Execution Framework Error: Unassigned cross-validation partitions found."

print("\n Cohort Class Balance Breakdown Across Extracted Folds:")
print(df.groupby(['fold', 'class_3']).size().unstack(fill_value=0).to_string())

# ─── 5. SERIALIZATION AND DETERMINISTIC REPRODUCIBILITY CHECKSUM ──────
csv_path = os.path.join(SPLIT_DIR, 'patient_folds.csv')
df.to_csv(csv_path, index=False)

# Compute rigid structural validation MD5 checksum for tracking logs
md5_checksum = hashlib.md5(df[['patient_id', 'fold']].to_csv(index=False).encode()).hexdigest()
print(f"\n✔ Dataset pipeline completed. Exported to → {csv_path}")
print(f"✔ Split Verification MD5 Checksum: {md5_checksum}")

Benign : Paired Patients Matched =  297
Malign : Paired Patients Matched =  367
Normal : Paired Patients Matched =  297
✔ Leakage Integrity Check: PASSED (Mutual exclusivity verified across classes)

 Cohort Class Balance Breakdown Across Extracted Folds:
class_3  Benign  Malign  Normal
fold                           
0            60      74      59
1            59      74      59
2            59      73      60
3            59      73      60
4            60      73      59

✔ Dataset pipeline completed. Exported to → ./splits/patient_folds.csv
✔ Split Verification MD5 Checksum: aa0ef8cf5f91f7e338ba5e989f8ab50b
